# Mnist Cnn Experiments

MNIST CNN experiments comparing training settings and regularization.

Stage: learning experiment.


Question 1

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import torch
from torch import nn

# Define the CNN model
class MNIST_CNN(nn.Module):
    def __init__(self):
        super(MNIST_CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)  # Input channels, output channels, kernel size
        self.pool = nn.MaxPool2d(2, 2)  # Pooling size, stride
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)  # Input features after convolutions, number of neurons
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)  # Output layer for 10 digits

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 16 * 4 * 4)  # Flatten for fully connected layers
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x


# Load MNIST data
train_data = datasets.MNIST('./data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.MNIST('./data', train=False, download=True, transform=transforms.ToTensor())

# Define hyperparameters (change these for evaluation)
epochs = 10
learning_rate = 0.001
batch_size = 64
loss_func = nn.CrossEntropyLoss()  # Example loss function, change for evaluation

# Train the model (modify this loop to evaluate different hyperparameters)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
model = MNIST_CNN()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)



for epoch in range(epochs):
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = loss_func(output, target)
        loss.backward()
        optimizer.step()

# Evaluate model accuracy
correct, total = 0, 0
with torch.no_grad():
    for data, target in test_loader:
        output = model(data)
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

print('Accuracy of the model on the test images: {} %'.format(100 * correct / total))

In [ ]:
def evaluate_epochs(epochs_list):
    for epochs in epochs_list:
        for epoch in range(epochs):
            for data, target in train_loader:
                optimizer.zero_grad()
                output = model(data)
                loss = loss_func(output, target)
                loss.backward()
                optimizer.step()
        correct, total = 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                output = model(data)
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()
        print('Epochs: {}, Accuracy: {} %'.format(epochs, 100 * correct / total))

def evaluate_learning_rate(lr_list):
    for lr in lr_list:
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        for epoch in range(epochs):
            for data, target in train_loader:
                optimizer.zero_grad()
                output = model(data)
                loss = loss_func(output, target)
                loss.backward()
                optimizer.step()
        correct, total = 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                output = model(data)
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()
        print('Learning Rate: {}, Accuracy: {} %'.format(lr, 100 * correct / total))

epoch_list = [5,10,15]
evaluate_epochs(epoch_list)

lr_list = [0.01,0.2,0.5,1.0,1.4]
evaluate_learning_rate(lr_list)

In [ ]:


def evaluate_optimizer(optimizer_list, model, train_loader, test_loader, epochs, loss_func):
    for optimizer_class in optimizer_list:
        lr = 0.001  # Default learning rate (you can adjust this)

        # Instantiate optimizer with the specified learning rate
        optimizer = optimizer_class(model.parameters(), lr=lr)

        # Training loop
        for epoch in range(epochs):
            model.train()  # Set model to training mode
            for data, target in train_loader:
                optimizer.zero_grad()
                output = model(data)
                loss = loss_func(output, target)
                loss.backward()
                optimizer.step()

        # Evaluation loop
        correct, total = 0, 0
        model.eval()  # Set model to evaluation mode (e.g., disables dropout)
        with torch.no_grad():
            for data, target in test_loader:
                output = model(data)
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

        # Print optimizer and accuracy
        print('Optimizer: {}, Learning Rate: {}, Accuracy: {:.2f} %'.format(optimizer.__class__.__name__, lr, 100 * correct / total))

# Example usage
model = MNIST_CNN()  # Assuming MNIST_CNN is defined as before
optimizer_list = [torch.optim.Adam, torch.optim.SGD, torch.optim.Adagrad]

evaluate_optimizer(optimizer_list, model, train_loader, test_loader, epochs=10, loss_func=nn.CrossEntropyLoss())


In [ ]:
def evaluate_loss_function(loss_func_list, model, optimizer, train_loader, test_loader, epochs):
    for loss_func in loss_func_list:
        for epoch in range(epochs):
            model.train()  # Set model to training mode
            for data, target in train_loader:
                optimizer.zero_grad()
                output = model(data)

                # For CrossEntropyLoss, target should be class labels (not one-hot encoded)
                # Reshape target to match the output shape (batch_size, num_classes)
                if isinstance(loss_func, nn.CrossEntropyLoss):
                    loss = loss_func(output, target)
                else:
                    # For other loss functions (e.g., MSE, BCE), reshape target if needed
                    # For example, if using BCELoss, ensure target has the same shape as output
                    target = target.float()  # Ensure target is float (for BCELoss)
                    loss = loss_func(output, target)

                loss.backward()
                optimizer.step()

        # Evaluation after training
        correct, total = 0, 0
        model.eval()  # Set model to evaluation mode
        with torch.no_grad():
            for data, target in test_loader:
                output = model(data)
                _, predicted = torch.max(output, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

        # Print results
        accuracy = 100 * correct / total
        print('Loss Function: {}, Accuracy: {:.2f} %'.format(loss_func.__class__.__name__, accuracy))

# Example usage
model = MNIST_CNN()  # Assuming MNIST_CNN is defined as before
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_func_list = [nn.CrossEntropyLoss()]

evaluate_loss_function(loss_func_list, model, optimizer, train_loader, test_loader, epochs=10)


2) Transfer Learning

In [ ]:
def train_and_evaluate(model_class, epochs=10, learning_rate=0.001, loss_func=nn.CrossEntropyLoss(), optimizer=torch.optim.Adam):
    """
    Trains and evaluates a model on MNIST dataset.

    Args:
        model_class (nn.Module): The model class to be trained.
        epochs (int, optional): Number of training epochs. Defaults to 10.
        learning_rate (float, optional): Learning rate for the optimizer. Defaults to 0.001.
        loss_func (nn.Module, optional): Loss function for training. Defaults to nn.CrossEntropyLoss().
        optimizer (torch.optim, optional): Optimizer to use for training. Defaults to torch.optim.Adam.
    """

    model = model_class()
    optimizer = optimizer(model.parameters(), lr=learning_rate)

    train_data = datasets.MNIST('./data', train=True, download=True, transform=transforms.ToTensor())
    test_data = datasets.MNIST('./data', train=False, download=True, transform=transforms.ToTensor())

    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

    for epoch in range(epochs):
        for data, target in train_loader:
            optimizer.zero_grad()
            output = model(data)
            loss = loss_func(output, target)
            loss.backward()
            optimizer.step()

    correct, total = 0, 0
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

    print(f'Accuracy of the model on the test images: {100 * correct / total:.2f} %')

train_and_evaluate(MNIST_CNN)

Regularization

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Define the CNN model
class MNIST_CNN(nn.Module):
    def __init__(self):
        super(MNIST_CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 16 * 4 * 4)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Load MNIST data
train_data = datasets.MNIST('./data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.MNIST('./data', train=False, download=True, transform=transforms.ToTensor())

# Define hyperparameters
epochs = 10
learning_rate = 0.001
batch_size = 64

# Initialize model, loss function, and optimizer
model = MNIST_CNN()
loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=0.001)  # Add weight decay (L2 regularization)

# Create data loaders
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

# Training loop
for epoch in range(epochs):
    model.train()  # Set model to training mode
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = loss_func(output, target)
        loss.backward()
        optimizer.step()

# Evaluate model accuracy
correct, total = 0, 0
model.eval()  # Set model to evaluation mode
with torch.no_grad():
    for data, target in test_loader:
        output = model(data)
        _, predicted = torch.max(output, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

# Calculate accuracy
accuracy = 100 * correct / total
print('Accuracy of the model on the test images: {:.2f} %'.format(accuracy))
